# Fine-tuning BioBART para Sumarizacao de Abstracts Biomedicos
## PubMed Summarization (`ccdv/pubmed-summarization`)

Notebook de treino para rodar no **Google Colab** (Runtime > Change runtime type > **GPU T4**).

Pipeline: carregar dataset -> pre-processar -> fine-tuning do `GanjinZero/biobart-v2-base`
com `Seq2SeqTrainer` -> avaliar **ROUGE** (fine-tuned **vs** baseline) -> publicar o modelo no
HuggingFace Hub para ser consumido pelo app Gradio.

> **Antes de comecar:** crie um token de escrita em https://huggingface.co/settings/tokens
> (role *Write*). Voce vai cola-lo na celula de login.

---
## 0 - Requisitos
Instala as dependencias. Execute apenas uma vez por sessao do Colab.

In [ ]:
%pip install -q -U "transformers>=4.44" datasets evaluate rouge_score accelerate huggingface_hub

---
## 1 - Login no HuggingFace Hub
Necessario para `push_to_hub`. Cole seu token (Write) quando solicitado.

In [ ]:
from huggingface_hub import notebook_login, whoami

notebook_login()

In [ ]:
HF_USER = whoami()['name']
print('Logado como:', HF_USER)

---
## 2 - Configuracao

Todos os "botoes" do experimento ficam aqui. Os limites `MAX_INPUT=1024` / `MAX_TARGET=256`
vem da configuracao **moderada recomendada na EDA** (`eda_pubmed.ipynb`).

> Se o tempo de treino apertar, reduza `TRAIN_SUBSET`, `EVAL_SUBSET` ou `EPOCHS`.

In [ ]:
import re
import numpy as np
import torch

MODEL_CKPT   = 'GanjinZero/biobart-v2-base'   # modelo base (BART pre-treinado em biomedicina)
MAX_INPUT    = 1024    # tokens do artigo (janela do BART base)
MAX_TARGET   = 256     # tokens do resumo gerado
TRAIN_SUBSET = 8000    # exemplos de treino (subconjunto, conforme o plano)
EVAL_SUBSET  = 500     # exemplos de teste para a avaliacao ROUGE
EPOCHS       = 2
LR           = 5e-5
BATCH        = 4       # por dispositivo; com grad-accum 2 => batch efetivo 8
NUM_BEAMS    = 4
SEED         = 42

MODEL_NAME   = 'biobart-pubmed-summarization'
HUB_MODEL_ID = f'{HF_USER}/{MODEL_NAME}'      # destino no Hub

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Modelo sera publicado em:', HUB_MODEL_ID)

---
## 3 - Carregar e limpar o dataset

Usamos um subconjunto do treino e subconjuntos de validacao/teste. Aplicamos os filtros
sugeridos na EDA: remover pares com abstract muito curto ou razao de compressao degenerada.

In [ ]:
from datasets import load_dataset

raw = load_dataset('ccdv/pubmed-summarization')
print(raw)

In [ ]:
def is_valid(ex):
    art = ex['article'].strip()
    abs_ = ex['abstract'].strip()
    if not art or not abs_:
        return False
    n_abs = len(abs_.split())
    n_art = len(art.split())
    if n_abs < 20:                 # abstracts muito curtos: provavel ruido
        return False
    ratio = n_art / max(n_abs, 1)  # razao de compressao
    if ratio < 2 or ratio > 100:   # pares degenerados / mal alinhados
        return False
    return True

train_ds = raw['train'].shuffle(seed=SEED).filter(is_valid).select(range(TRAIN_SUBSET))
val_ds   = raw['validation'].filter(is_valid).select(range(min(500, len(raw['validation']))))
test_ds  = raw['test'].filter(is_valid).select(range(EVAL_SUBSET))

print(f'Treino : {len(train_ds):,}')
print(f'Val    : {len(val_ds):,}')
print(f'Teste  : {len(test_ds):,}')

---
## 4 - Tokenizacao

O artigo vira a entrada (truncado em `MAX_INPUT`) e o abstract vira o alvo (`MAX_TARGET`).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def preprocess(batch):
    model_inputs = tokenizer(batch['article'], max_length=MAX_INPUT, truncation=True)
    labels = tokenizer(text_target=batch['abstract'], max_length=MAX_TARGET, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

cols = train_ds.column_names
tok_train = train_ds.map(preprocess, batched=True, remove_columns=cols, desc='Tokenizando treino')
tok_val   = val_ds.map(preprocess,   batched=True, remove_columns=cols, desc='Tokenizando val')
print('OK')

---
## 5 - Modelo, collator e metrica ROUGE

In [ ]:
import evaluate
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge = evaluate.load('rouge')

def split_sents(text):
    # rougeLsum funciona melhor com uma sentenca por linha (sem dependencia do nltk)
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return '\n'.join(s.strip() for s in parts if s.strip())

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    dec_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    dec_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    dec_preds  = [split_sents(p) for p in dec_preds]
    dec_labels = [split_sents(l) for l in dec_labels]
    result = rouge.compute(predictions=dec_preds, references=dec_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

---
## 6 - Treino (`Seq2SeqTrainer`)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir=MODEL_NAME,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.05,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    generation_num_beams=NUM_BEAMS,
    fp16=torch.cuda.is_available(),
    report_to='none',
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,  # transformers v5: substitui o antigo tokenizer=
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

---
## 7 - Avaliacao: fine-tuned **vs** baseline

Comparamos o modelo ajustado com o `biobart-v2-base` **sem fine-tuning** no mesmo subconjunto
de teste (objetivos (i) e (iii) do plano). A mesma funcao gera os resumos para os dois modelos,
garantindo uma comparacao justa.

In [ ]:
@torch.no_grad()
def evaluate_model(eval_model, dataset, batch_size=8):
    eval_model = eval_model.to(device).eval()
    preds, refs = [], []
    for i in range(0, len(dataset), batch_size):
        chunk = dataset[i:i + batch_size]
        inputs = tokenizer(chunk['article'], max_length=MAX_INPUT, truncation=True,
                           padding=True, return_tensors='pt').to(device)
        out = eval_model.generate(**inputs, max_length=MAX_TARGET, num_beams=NUM_BEAMS)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        refs.extend(chunk['abstract'])
    dp = [split_sents(p) for p in preds]
    dr = [split_sents(r) for r in refs]
    scores = rouge.compute(predictions=dp, references=dr, use_stemmer=True)
    scores = {k: round(v * 100, 2) for k, v in scores.items()}
    return scores, preds, refs

In [ ]:
print('Avaliando modelo FINE-TUNED no teste...')
ft_scores, ft_preds, refs = evaluate_model(trainer.model, test_ds)
print(ft_scores)

In [ ]:
print('Avaliando BASELINE (biobart-v2-base sem fine-tuning)...')
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)
bl_scores, bl_preds, _ = evaluate_model(baseline_model, test_ds)
print(bl_scores)

In [ ]:
import pandas as pd

comp = pd.DataFrame({
    'Baseline (sem FT)': bl_scores,
    'Fine-tuned':        ft_scores,
}).T
comp['delta_rouge1'] = (comp['rouge1'] - bl_scores['rouge1']).round(2)
print('=== Comparacao ROUGE (subconjunto de teste, n=%d) ===' % len(test_ds))
display(comp)
print('\nCopie esta tabela para o relatorio_final.md')

---
## 8 - Amostras qualitativas

Inspecao manual (objetivo (ii) do plano): artigo -> resumo gerado x resumo de referencia.

In [ ]:
for i in range(3):
    print('=' * 90)
    print(f'EXEMPLO #{i}')
    print('=' * 90)
    print('ARTIGO (inicio):\n', test_ds[i]['article'][:500], '...\n')
    print('--- RESUMO GERADO (fine-tuned) ---\n', ft_preds[i], '\n')
    print('--- RESUMO BASELINE (sem FT) ---\n', bl_preds[i], '\n')
    print('--- ABSTRACT DE REFERENCIA ---\n', test_ds[i]['abstract'], '\n')

---
## 9 - Publicar o modelo no Hub

Envia pesos + tokenizer + model card para `HUB_MODEL_ID`. Depois disso, o app Gradio
(em `app/app.py`) consegue carregar o modelo so com esse id.

In [ ]:
trainer.push_to_hub(commit_message='Fine-tuned BioBART em PubMed summarization')
tokenizer.push_to_hub(HUB_MODEL_ID)
print('Modelo publicado em: https://huggingface.co/' + HUB_MODEL_ID)

---
### Proximos passos
1. Anote o `HUB_MODEL_ID` impresso acima e os numeros de ROUGE da tabela.
2. No `app/app.py`, defina `MODEL_ID` com esse id e suba o conteudo de `app/` no HuggingFace Space.
3. Preencha a tabela de resultados em `relatorio/relatorio_final.md`.